In [9]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d
import pymeshlab
import trimesh

from plant_shape_analysis.dataloaders.trackplant3D import LeafSequencesDataset
from plant_shape_analysis.data_preprocessing.leaf_meshing import mesh_leaf

from plant_shape_analysis.reconstruction import classical_methods

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Mesh example single leaf

In [10]:
dataset_path = Path("../data/TrackPlant3D")
leaf_dataset = LeafSequencesDataset(
    dataset_path, 
    min_timepoints=1, # we get all leaves
)

In [11]:
maize_leaves = leaf_dataset.get_timeseries_by_crop('maize')
len(maize_leaves)

13

In [12]:
seq_id = 0
timepoint = 0
leaf = maize_leaves[seq_id]['timepoints'][timepoint]
leaf

{'day': 0,
 'leaf_id': 1,
 'points': array([[ 6.528425, -4.4796  ,  3.937625],
        [ 6.42105 , -8.4412  , 20.428499],
        [-2.45315 , -4.028133, 12.709   ],
        ...,
        [ 7.125534, -6.243192,  7.31445 ],
        [-0.10225 , -2.6813  ,  3.67935 ],
        [-5.865   ,  1.13345 , -2.873725]], shape=(5594, 3)),
 'dense_points': array([[ -4.184584,   6.560559, -12.768311],
        [ -4.346184,   6.494059, -12.859811],
        [ -4.381384,   6.433059, -12.925611],
        ...,
        [ -5.713584,   3.374759, -13.906611],
        [ -5.626284,   3.374859, -13.941411],
        [ -5.537884,   3.372259, -13.974911]], shape=(155504, 3)),
 'leaf_tip': array([ 7.34715 , -9.7354  , 19.847799]),
 'file_path': PosixPath('../data/TrackPlant3D/gt_corrected_v1/maize/1_maize_control_plant1_D00.txt')}

In [13]:
pcd, mesh = mesh_leaf(leaf['points'], method="ball_pivoting", backend="pymeshlab")
mesh.get_surface_area()

840.1121893577159

In [14]:
pcd, mesh = mesh_leaf(leaf['points'], method="ball_pivoting", backend="open3d")
mesh.get_surface_area()

470.4855310811755

In [15]:
pcd, mesh = mesh_leaf(leaf['points'], method="poisson", backend="pymeshlab")
mesh.get_surface_area()

1767.0846451336686

In [16]:
pcd, mesh = mesh_leaf(leaf['points'], method="poisson", backend="open3d")
mesh.get_surface_area()

1751.167697639144

In [ ]:
o3d.visualization.draw_geometries(
[mesh],
point_show_normal=True,
window_name="Leaf Point Cloud and normals",
)

## Extract all dense leaves

In [ ]:
dataset_path = Path("../data/TrackPlant3D")
leaf_dataset = LeafSequencesDataset(
    dataset_path, 
    min_timepoints=1, # we get all leaves
)

In [ ]:
# for seq in leaf_dataset:
#     for scan in seq['timepoints']:
#         if scan['dense_points'] is not None:
#             # save leaf
#             crop = ''.join([c for c in seq['sequence_name'].split('_')[0] if c.isalpha()])
#             file_name = scan['file_path'].name.replace(".txt", f"_leaf{scan['leaf_id']}_dense.ply")
#             pcd = o3d.geometry.PointCloud()
#             pcd.points = o3d.utility.Vector3dVector(scan['dense_points'])
#             save_dir = Path(f"../data/TrackPlant3D/dense_leaves/{crop}")
#             save_dir.mkdir(parents=True, exist_ok=True)
#             o3d.io.write_point_cloud(save_dir / file_name, pcd)

## Classical methods for surface reconstruction

In [9]:
# Load and visualize a sample leaf point cloud
#sample_leaf_path = Path("../data/TrackPlant3D/dense_leaves/maize/19_maize_control_plant3_D00_leaf1_dense.ply")
sample_leaf_path = Path("../data/TrackPlant3D/examples/single_leafs/maize/1_maize_control_plant1_D00_leaf1_dense.ply")
#sample_leaf_path = Path("../data/TrackPlant3D/examples/single_leafs/maize/1_maize_control_plant1_D00_leaf1_dense_downsampled-voxel02.ply")
leaf_pcd = o3d.io.read_point_cloud(str(sample_leaf_path))
leaf_np = np.asarray(leaf_pcd.points)
print(f"Loaded leaf point cloud with {leaf_np.shape[0]} points.")

# Get normals too 
leaf_pcd.estimate_normals()
leaf_pcd.orient_normals_to_align_with_direction()
leaf_pcd.orient_normals_consistent_tangent_plane(k=50)
point_cloud = np.hstack([leaf_np, np.asarray(leaf_pcd.normals)])

o3d.visualization.draw_geometries([leaf_pcd])

Loaded leaf point cloud with 155504 points.


### Apha shapes

In [8]:
mesh_alpha = classical_methods.alpha_shape_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_alpha]) # -> Not bad but has holes
mesh_alpha.get_surface_area()

953.550073585376

### Bezier

In [ ]:
mesh_bezier = classical_methods.bezier_surface_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_bezier]) # -> Has some issues, needs tuning
mesh_bezier.get_surface_area()

### B-spline

In [ ]:
mesh_bspline = classical_methods.bspline_surface_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_bspline]) # -> Looks better than bezier, but has some singularities
mesh_bspline.get_surface_area()

### Delaunay

In [ ]:
mesh_delaunay = classical_methods.delaunay_mesh_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_delaunay])
mesh_delaunay.get_surface_area()

### Poisson

In [18]:
mesh_poisson = classical_methods.poisson_reconstruction(leaf_np, depth=9)
o3d.visualization.draw_geometries([mesh_poisson])
mesh_poisson.get_surface_area()

1783.87154623626

### Ball pivoting

In [ ]:
mesh_bpa = classical_methods.ball_pivoting_reconstruction(leaf_np, radius_multiplier=5.0)
o3d.visualization.draw_geometries([mesh_bpa])
mesh_bpa.get_surface_area()

Alternatively the area could be computed as below but it gives the same results

In [ ]:
vertices_bezier = np.asarray(mesh_bezier.vertices)
triangles_bezier = np.asarray(mesh_bezier.triangles)

area_bezier_manual = classical_methods.compute_mesh_area(vertices_bezier, triangles_bezier)
float(area_bezier_manual)

## Trimmed Poisson

In [26]:
# Load and visualize a sample leaf point cloud
#sample_leaf_path = Path("../data/TrackPlant3D/dense_leaves/maize/19_maize_control_plant3_D00_leaf1_dense.ply")
sample_leaf_path = Path("../data/TrackPlant3D/examples/single_leafs/maize/1_maize_control_plant1_D00_leaf1_dense.ply")
#sample_leaf_path = Path("../data/TrackPlant3D/examples/single_leafs/maize/1_maize_control_plant1_D00_leaf1_dense_downsampled-voxel02.ply")
leaf_pcd = o3d.io.read_point_cloud(str(sample_leaf_path))

leaf_pcd.estimate_normals()
leaf_pcd.orient_normals_to_align_with_direction()
leaf_pcd.orient_normals_consistent_tangent_plane(k=50)
leaf_np = np.asarray(leaf_pcd.points)
point_cloud = np.hstack([leaf_np, np.asarray(leaf_pcd.normals)])
print(f"Loaded leaf point cloud with {leaf_np.shape[0]} points.")
o3d.visualization.draw_geometries([leaf_pcd])

Loaded leaf point cloud with 155504 points.


### Reconstruct Poission surface

In [ ]:
#pcd, mesh = mesh_leaf(leaf_np, method="poisson", backend="pymeshlab")

With Meshlab

In [ ]:
# ms = pymeshlab.MeshSet()
# mesh = pymeshlab.Mesh(vertex_matrix=leaf_np)
# ms.add_mesh(mesh, "leaf")
# ms.compute_normal_for_point_clouds(k=20, smoothiter=10)
# ms.generate_surface_reconstruction_screened_poisson(depth=7)
# mesh = ms.current_mesh()

# vertices = np.asarray(mesh.vertex_matrix())
# triangles = np.asarray(mesh.face_matrix())

# o3d_mesh = o3d.geometry.TriangleMesh()
# o3d_mesh.vertices = o3d.utility.Vector3dVector(vertices)
# o3d_mesh.triangles = o3d.utility.Vector3iVector(triangles)
# o3d_mesh.compute_vertex_normals() # this makes faces look good
# o3d.visualization.draw_geometries([o3d_mesh])

Try poisson with open3d, and try to filter out points based on density

In [3]:
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(leaf_np)

# Estimate normals
pcd.estimate_normals()
pcd.orient_normals_to_align_with_direction()
pcd.orient_normals_consistent_tangent_plane(k=30)  # This is very important!!

# Create mesh using Poisson Surface Reconstruction
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=7, scale=1.1)
mesh.compute_vertex_normals()
mesh.vertex_colors = o3d.utility.Vector3dVector(np.empty((0, 3)))
o3d.visualization.draw_geometries([mesh])

In [4]:
densities = np.asarray(densities)
density_colors = plt.cm.plasma(densities / np.max(densities))[:, :3]  # Use colormap
mesh.vertex_colors = o3d.utility.Vector3dVector(density_colors)
o3d.visualization.draw_geometries([mesh])

### Based on density only

In [ ]:
def apply_threshold_and_visualize(mesh, densities, threshold):
    """Apply threshold and show before/after comparison"""
    
    # Get original data
    vertices = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.triangles)
    
    # Create mask for vertices to keep
    vertices_to_keep = densities >= threshold
    
    # Create mapping from old indices to new indices
    vertex_mapping = np.full(len(vertices), -1)
    vertex_mapping[vertices_to_keep] = np.arange(np.sum(vertices_to_keep))
    
    # Filter vertices
    new_vertices = vertices[vertices_to_keep]
    
    # Filter triangles - keep only triangles where all vertices are kept
    triangle_mask = np.all(vertices_to_keep[triangles], axis=1)
    new_triangles = triangles[triangle_mask]
    
    # Remap triangle indices
    new_triangles = vertex_mapping[new_triangles]
    
    # Create new mesh
    trimmed_mesh = o3d.geometry.TriangleMesh()
    trimmed_mesh.vertices = o3d.utility.Vector3dVector(new_vertices)
    trimmed_mesh.triangles = o3d.utility.Vector3iVector(new_triangles)
    trimmed_mesh.compute_vertex_normals()

    
    # Copy colors if they exist
    if mesh.has_vertex_colors():
        vertex_colors = np.asarray(mesh.vertex_colors)
        new_vertex_colors = vertex_colors[vertices_to_keep]
        trimmed_mesh.vertex_colors = o3d.utility.Vector3dVector(new_vertex_colors)
    
    # Show statistics
    original_vertices = len(vertices)
    remaining_vertices = len(new_vertices)
    print(f"Original vertices: {original_vertices}")
    print(f"Remaining vertices: {remaining_vertices}")
    print(f"Removed: {original_vertices - remaining_vertices} ({100*(1-remaining_vertices/original_vertices):.1f}%)")
    
    # Visualize both
    print("Original mesh (press Q to close and see trimmed)")
    o3d.visualization.draw_geometries([mesh], window_name="Original Mesh")
    
    print("Trimmed mesh")
    o3d.visualization.draw_geometries([trimmed_mesh], window_name="Trimmed Mesh")
    
    return trimmed_mesh

In [ ]:
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(leaf_np)

# Estimate normals
pcd.estimate_normals()
pcd.orient_normals_to_align_with_direction()
pcd.orient_normals_consistent_tangent_plane(k=30)  # This is very important!!

# Create mesh using Poisson Surface Reconstruction
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=7, scale=1.1)
mesh.compute_vertex_normals()
mesh.vertex_colors = o3d.utility.Vector3dVector(np.empty((0, 3)))
o3d.visualization.draw_geometries([mesh])

In [ ]:
selected_threshold = np.percentile(densities, 10)  # Adjust based on your plots
trimmed_mesh = apply_threshold_and_visualize(mesh, densities, selected_threshold)
# o3d.io.write_triangle_mesh("trimmed_leaf_mesh.ply", trimmed_mesh)
# pcd, mesh = mesh_leaf(leaf_np, method="poisson", backend="pymeshlab")

### Density + distance based

In [ ]:
def create_alpha_shape_boundary(pcd, alpha=0.1):
    """Create alpha shape to define better boundaries"""
    
    # Create alpha shape (2D or 3D)
    alpha_shape = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd, alpha)
    alpha_shape.compute_vertex_normals()
    
    return alpha_shape

def combine_poisson_with_alpha_boundary(pcd, poisson_depth=7, alpha=0.1, density_threshold_percentile=10):
    """Combine Poisson reconstruction with alpha shape boundary"""
    
    # Step 1: Poisson reconstruction
    poisson_mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd, depth=poisson_depth
    )
    
    # Step 2: Create alpha shape for boundary reference
    alpha_shape = create_alpha_shape_boundary(pcd, alpha)
    
    # Step 3: Filter Poisson mesh by density
    density_threshold = np.percentile(densities, density_threshold_percentile)
    vertices_to_keep = densities >= density_threshold
    
    # Step 4: Additional filtering - keep vertices close to alpha shape
    poisson_vertices = np.asarray(poisson_mesh.vertices)
    alpha_vertices = np.asarray(alpha_shape.vertices)
    
    # Find distances from Poisson vertices to alpha shape vertices
    from scipy.spatial.distance import cdist
    distances = cdist(poisson_vertices, alpha_vertices)
    min_distances = np.min(distances, axis=1)
    
    # Additional mask: keep vertices close to alpha shape
    distance_threshold = np.percentile(min_distances, 80)  # Keep 80% closest
    distance_mask = min_distances <= distance_threshold
    
    # Combine both masks
    final_mask = vertices_to_keep & distance_mask
    
    # Create filtered mesh
    trimmed_mesh = filter_mesh_by_vertex_mask(poisson_mesh, final_mask)
    
    return trimmed_mesh, alpha_shape, poisson_mesh

def filter_mesh_by_vertex_mask(mesh, vertex_mask):
    """Filter mesh by vertex mask"""
    vertices = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.triangles)
    
    # Create vertex mapping
    vertex_mapping = np.full(len(vertices), -1)
    vertex_mapping[vertex_mask] = np.arange(np.sum(vertex_mask))
    
    # Filter vertices
    new_vertices = vertices[vertex_mask]
    
    # Filter triangles
    triangle_mask = np.all(vertex_mask[triangles], axis=1)
    new_triangles = triangles[triangle_mask]
    new_triangles = vertex_mapping[new_triangles]
    
    # Create new mesh
    filtered_mesh = o3d.geometry.TriangleMesh()
    filtered_mesh.vertices = o3d.utility.Vector3dVector(new_vertices)
    filtered_mesh.triangles = o3d.utility.Vector3iVector(new_triangles)
    filtered_mesh.compute_vertex_normals()
    
    return filtered_mesh

In [41]:
# Check out alpha shape
trimmed_mesh, alpha_shape, poission_mesh = combine_poisson_with_alpha_boundary(pcd, alpha=0.6)
o3d.visualization.draw_geometries([pcd, trimmed_mesh])

In [43]:
trimmed_mesh.get_surface_area()

515.039270475723

In [ ]:
def improve_mesh_boundary_simple(pcd, poisson_depth=9, density_percentile=15, distance_multiplier=2.0):
    """
    Improve Poisson mesh boundary by filtering vertices that are too far from original points
    """
    
    # Step 1: Poisson reconstruction
    print("Creating Poisson mesh...")
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd, depth=poisson_depth
    )
    
    # Step 2: Filter by density
    density_threshold = np.percentile(densities, density_percentile)
    density_mask = densities >= density_threshold
    print(f"Density filtering keeps: {np.sum(density_mask)}/{len(density_mask)} vertices")
    
    # Step 3: Calculate distance threshold based on point cloud spacing
    pcd_points = np.asarray(pcd.points)
    from sklearn.neighbors import NearestNeighbors
    
    # Find average spacing in point cloud
    nbrs = NearestNeighbors(n_neighbors=5, algorithm='kd_tree').fit(pcd_points)
    distances_pcd, _ = nbrs.kneighbors(pcd_points)
    avg_spacing = np.mean(distances_pcd[:, 1])
    distance_threshold = distance_multiplier * avg_spacing
    
    print(f"Average point cloud spacing: {avg_spacing:.6f}")
    print(f"Using distance threshold: {distance_threshold:.6f}")
    
    # Step 4: Filter mesh vertices by distance to original points
    mesh_vertices = np.asarray(mesh.vertices)
    nbrs_mesh = NearestNeighbors(n_neighbors=1, algorithm='kd_tree').fit(pcd_points)
    distances_to_pcd, _ = nbrs_mesh.kneighbors(mesh_vertices)
    distances_to_pcd = distances_to_pcd.flatten()
    
    distance_mask = distances_to_pcd <= distance_threshold
    print(f"Distance filtering keeps: {np.sum(distance_mask)}/{len(distance_mask)} vertices")
    
    # Step 5: Combine filters and remove vertices
    vertices_to_remove = ~(density_mask & distance_mask)
    print(f"Removing {np.sum(vertices_to_remove)}/{len(vertices_to_remove)} vertices")
    
    # Step 6: Apply filtering
    mesh.remove_vertices_by_mask(vertices_to_remove)
    mesh.compute_vertex_normals()
    
    return mesh, distances_to_pcd

In [ ]:
improved_mesh, original_mesh, distances = improve_mesh_boundary_simple(
    pcd, 
    poisson_depth=7,
    density_percentile=10,
    distance_multiplier=10.0  # Adjust this to make boundary tighter (smaller) or looser (larger)
)

# Visualize
print("Original Poisson mesh")
o3d.visualization.draw_geometries([pcd, original_mesh])

print("Improved mesh (closer to point cloud boundary)")
o3d.visualization.draw_geometries([improved_mesh])

# Optional: visualize with point cloud overlay
print("Improved mesh with original points")
o3d.visualization.draw_geometries([pcd, improved_mesh])

Creating Poisson mesh...
Density filtering keeps: 32621/36246 vertices
Average point cloud spacing: 0.046487
Using distance threshold: 0.464865
Distance filtering keeps: 32714/36246 vertices
Combined filtering keeps: 32436/36246 vertices
Original Poisson mesh
Improved mesh (closer to point cloud boundary)
Improved mesh with original points


In [30]:
improved_mesh.get_surface_area() # This seems to give the most reasonable visual results and plausible area

495.3601458728539